# 第 6 章｜异步子 Agent 的完整生命周期

对应 [第 6 章正文](../../content/ch06-async-subagents.md)。本实验从 Notebook 启动真实本地 Agent Server，观察 supervisor 通过 ASGI 委派后的后台任务。

**学习目标**：说明主任务返回与后台任务完成的区别；使用五个异步工具；通过 task/thread/run ID 和真实服务状态验证完成、更新与取消。

**预期现象**：启动返回时子任务尚在运行；更新保留 task ID 并创建新 run；取消后主 Agent 记录 cancelled，服务端 run 进入 interrupted。

## 1. 环境与模式

按 [统一安装说明](../README.md) 使用 Python 3.12 和 server 依赖组：

```bash
uv sync --project notebooks --locked --extra server
uv run --project notebooks --locked --extra server python -m course_notebooks.run ch06-async-subagents
```

默认 offline 用脚本模型规定 supervisor 选择哪个工具，服务、中间件、SDK 和后台图都真实执行。随附输出来自本次 offline 运行；它不证明真实模型的选工具能力。切换真实模型需按 README 配置并显式添加 `--mode live`，可能产生模型费用。无需远程部署，也不要求 LangSmith Key。

In [1]:
import asyncio
import json
import sys
import time

from course_notebooks.model_config import repository_root
from course_notebooks.nbtools import show_runtime, show_text

ROOT = repository_root()
CHAPTER_DIR = ROOT / "notebooks/ch06"
sys.path.insert(0, str(CHAPTER_DIR))
from local_server import LocalAgentServer

show_runtime()

运行模式： offline （脚本模型）
Python： 3.12.13 平台： Darwin arm64
deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2


## 2. Notebook、服务与两个图是什么关系？

```text
Notebook（SDK 客户端）
  └─ 本地 Agent Server
       ├─ supervisor：主 Agent，选择异步工具
       └─ researcher：后台图，等待后返回结果
            ↑ AsyncSubAgent 通过同部署的 ASGI 路径调用
```

| 名称 | 本例含义 | 更新任务时 |
|---|---|---|
| graph_id | 服务器中注册的图，如 researcher | 保持不变 |
| thread_id | 一段有状态的会话 | 子 thread 不变 |
| task_id | 主 Agent 跟踪的子任务 ID，本例等于子 thread_id | 保持不变 |
| run_id | 在该 thread 上的一次执行 | 创建新的 run |

`async def` 定义可等待的函数，`await` 在等待网络或后台结果时让出执行权。真正的后台任务由 Agent Server 调度，不是 Notebook 自己创建一个 Python await 就模拟出来的。

服务注册见 [langgraph.json](langgraph.json)，主图见 [supervisor.py](graphs/supervisor.py)。下面直接展示当前文件中的核心构造，避免维护第二份实现。

In [2]:
from IPython.display import Code, display

supervisor_source = (CHAPTER_DIR / "graphs/supervisor.py").read_text()
display(Code("graph = " + supervisor_source.split("graph = ", 1)[1], language="python"))

graph = create_deep_agent(
    model=model,
    system_prompt=(
        "This is an async-subagent tool experiment. For each user request, call exactly the "
        "requested async task tool once, then stop. Use researcher for starts. Never invent a "
        "task ID or report a cached status as live."
    ),
    subagents=[
        AsyncSubAgent(
            name="researcher",
            description="A slow local research graph for observing background tasks.",
            graph_id="researcher",
        )
    ],
)

### 一个可观察的后台图

researcher 故意等待 8 秒后回传输入，给查询、更新和取消留出窗口。它是固定教学图，不进行真实研究，也不用于比较研究质量或远程部署性能。

In [3]:
display(Code((CHAPTER_DIR / "graphs/researcher.py").read_text(), language="python"))

"""A deliberately slow graph so background execution is observable."""

import asyncio

from langgraph.graph import END, START, MessagesState, StateGraph


async def research(state: MessagesState):
    request = next(
        (message.content for message in reversed(state["messages"]) if message.type == "human"),
        "",
    )
    await asyncio.sleep(8)
    return {"messages": [{"role": "ai", "content": f"Research completed: {request}"}]}


builder = StateGraph(MessagesState)
builder.add_node("research", research)
builder.add_edge(START, "research")
builder.add_edge("research", END)
graph = builder.compile()

## 3. 如何检查一次工具调用？

`ask` 向服务端 supervisor 发送消息，再读取本轮的真实工具调用和返回。脚本模式使用 `START|说明`、`CHECK|task_id` 等明确命令；live 模式下由真实模型理解同一请求。

返回 state 是服务端运行后的状态。我们检查实际调用了预期工具，不能只相信模型说“已完成”。轮询只在状态改变时打印，并设置截止时间。

In [4]:
async def ask(server, text, expected_tool, show=True):
    state = await asyncio.wait_for(server.client.runs.wait(
        server.parent_id, "supervisor", input={"messages": [{"role": "user", "content": text}]}
    ), timeout=30)
    last_user = max(i for i, msg in enumerate(state["messages"]) if msg["type"] == "human")
    recent = state["messages"][last_user:]
    calls = [call["name"] for msg in recent for call in msg.get("tool_calls", [])]
    outputs = [msg for msg in recent if msg["type"] == "tool"]
    assert calls == [expected_tool], f"工具不符：{calls}"
    assert len(outputs) == 1 and outputs[0].get("status", "success") == "success", "工具调用失败。"
    server.task_ids.update(state.get("async_tasks", {}))
    output = outputs[0]["content"]
    if show:
        show_text(expected_tool, output)
    return state, output


async def wait_task(server, task_id, timeout=40):
    deadline, last_status = time.monotonic() + timeout, None
    while time.monotonic() < deadline:
        state, output = await ask(server, f"CHECK|{task_id}", "check_async_task", show=False)
        status = state["async_tasks"][task_id]["status"]
        if status != last_status:
            show_text("check_async_task", output)
            last_status = status
        if status == "success":
            return state, json.loads(output)
        if status in {"error", "cancelled", "interrupted", "timeout"}:
            raise RuntimeError(f"子任务异常结束：{status}")
        await asyncio.sleep(1)
    raise TimeoutError(f"任务未在 {timeout}s 内完成")

## 4. 启动、列举并等待完成

启动应立即给出 task ID。随后读服务端 run 状态，证明后台任务仍在执行；查询成功后核对实际结果。先定义每个阶段，最后在同一生命周期中执行，确保任何阶段失败都能清理。

In [5]:
async def observe_completion(server):
    print("阶段一：启动 → 列举 → 完成")
    state, output = await ask(server, "START|总结异步任务的状态变化", "start_async_task")
    task_id = next(iter(state["async_tasks"]))
    task = state["async_tasks"][task_id]
    assert task_id in output and task["status"] == "running"
    run = await server.client.runs.get(task_id, task["run_id"])
    assert run["status"] in {"pending", "running"}, run["status"]
    print("主 Agent 已返回，后台 run 仍为：", run["status"])
    _, listing = await ask(server, "LIST", "list_async_tasks")
    assert task_id in listing
    state, result = await wait_task(server, task_id)
    assert "Research completed" in result["result"]
    run = await server.client.runs.get(task_id, state["async_tasks"][task_id]["run_id"])
    assert run["status"] == "success"
    return {"阶段": "完成", "task_id": task_id, "服务状态": run["status"]}

## 5. 更新：同一个 task，新的 run

另起一个任务，再追加要求。验证 task ID 没变、run ID 已变，并核对实际发送的追加指令进入了子图结果。真实模型可以改写措辞，因此检查工具实际参数，不要求逐字复述用户提示。

In [6]:
async def observe_update(server):
    print("\n阶段二：更新 → 新 run → 读取新结果")
    previous = set(server.task_ids)
    state, _ = await ask(server, "START|撰写一份简短摘要", "start_async_task")
    task_id = (set(state["async_tasks"]) - previous).pop()
    old_run = state["async_tasks"][task_id]["run_id"]
    state, output = await ask(server, f"UPDATE|{task_id}|追加要求：使用三条要点", "update_async_task")
    updated = state["async_tasks"][task_id]
    assert task_id in output and updated["run_id"] != old_run
    instructions = [call["args"]["message"] for msg in state["messages"]
                    for call in msg.get("tool_calls", []) if call["name"] == "update_async_task"]
    assert instructions and instructions[-1].strip()
    state, result = await wait_task(server, task_id)
    assert instructions[-1] in result["result"], "追加指令没有进入子图结果。"
    run = await server.client.runs.get(task_id, updated["run_id"])
    assert run["status"] == "success"
    print("已验证：task ID 不变、run ID 更新、追加指令传递成功。")
    return {"阶段": "更新", "task_id": task_id, "服务状态": run["status"]}

## 6. 取消：核对主 Agent 和服务端两侧

取消第三个任务后，主 Agent 的跟踪状态为 cancelled；服务端确认 run 进入 interrupted 或 cancelled 后才算验证完成。

In [7]:
async def observe_cancellation(server):
    print("\n阶段三：取消 → 服务端确认")
    previous = set(server.task_ids)
    state, _ = await ask(server, "START|准备一个将被取消的任务", "start_async_task")
    task_id = (set(state["async_tasks"]) - previous).pop()
    state, output = await ask(server, f"CANCEL|{task_id}", "cancel_async_task")
    task = state["async_tasks"][task_id]
    assert task_id in output and task["status"] == "cancelled"
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        run = await server.client.runs.get(task_id, task["run_id"])
        if run["status"] in {"interrupted", "cancelled"}:
            break
        await asyncio.sleep(0.25)
    else:
        raise TimeoutError("服务端未确认取消")
    _, listing = await ask(server, "LIST", "list_async_tasks")
    assert all(task_id in listing for task_id in server.task_ids)
    return {"阶段": "取消", "task_id": task_id, "服务状态": run["status"]}

## 7. 在一个受保护的生命周期内运行

`async with` 进入时启动服务并等待就绪；正常完成或抛出异常，退出时都会清理。辅助代码只管理本次创建的资源，见 [local_server.py](local_server.py)。核心工具调用与断言仍在上面的阶段函数中。

下面的输出按三个阶段排列。只绑定本机地址；所有实验状态在临时服务目录，不污染仓库。

In [8]:
async with LocalAgentServer(CHAPTER_DIR, ROOT) as server:
    observations = [
        await observe_completion(server),
        await observe_update(server),
        await observe_cancellation(server),
    ]

print("\n生命周期验证结果：")
for observation in observations:
    print(observation["阶段"], "→", observation["服务状态"])

Agent Server 已就绪（仅本机访问）；已创建本次主 thread。
阶段一：启动 → 列举 → 完成



start_async_task
Launched async subagent. task_id: 01a0dcc5-6c72-7392-aae9-d6cdb53f4ff0
主 Agent 已返回，后台 run 仍为： pending



list_async_tasks
1 tracked task(s):
- task_id: 01a0dcc5-6c72-7392-aae9-d6cdb53f4ff0  agent: researcher  status:
  running



check_async_task
{"status": "running", "thread_id": "01a0dcc5-6c72-7392-aae9-d6cdb53f4ff0"}



check_async_task
{"status": "success", "thread_id": "01a0dcc5-6c72-7392-aae9-d6cdb53f4ff0",
  "result": "Research completed:
  \u603b\u7ed3\u5f02\u6b65\u4efb\u52a1\u7684\u72b6\u6001\u53d8\u5316"}

阶段二：更新 → 新 run → 读取新结果


...


check_async_task
{"status": "running", "thread_id": "01a0dcc5-978e-7881-a450-331f15615e09"}



check_async_task
{"status": "success", "thread_id": "01a0dcc5-978e-7881-a450-331f15615e09",
  "result": "Research completed:
  \u8ffd\u52a0\u8981\u6c42\uff1a\u4f7f\u7528\u4e09\u6761\u8981\u70b9"}
已验证：task ID 不变、run ID 更新、追加指令传递成功。

阶段三：取消 → 服务端确认



start_async_task
Launched async subagent. task_id: 01a0dcc5-c2ab-7611-9d51-9c577379d71e



cancel_async_task
Cancelled async subagent task: 01a0dcc5-c2ab-7611-9d51-9c577379d71e



list_async_tasks
3 tracked task(s):
- task_id: 01a0dcc5-6c72-7392-aae9-d6cdb53f4ff0  agent: researcher  status:
  success
- task_id: 01a0dcc5-978e-7881-a450-331f15615e09  agent: researcher  status:
  success
- task_id: 01a0dcc5-c2ab-7611-9d51-9c577379d71e  agent: researcher  status:
  cancelled
本次创建的任务、thread、服务进程与临时状态已清理。

生命周期验证结果：
完成 → success
更新 → success
取消 → interrupted


## 观察结论、练习与故障定位

- 五个工具通过真实 supervisor 调用；task ID 对应子 thread，run ID 标识一次执行。
- 更新保留 task ID 并创建新 run；取消同时检查主 Agent 状态和服务端终态。
- 本例只验证本地服务、ASGI 与生命周期；脚本模型不验证真实模型选择工具，固定 researcher 不验证研究质量。
- **练习**：在最后一格首个阶段之后主动 `raise ValueError("练习：提前退出")`，观察清理仍发生，失败日志保留；恢复后重跑。章内回归测试也覆盖后台任务运行时失败的清理。
- 找不到 CLI 时安装 server 依赖组；启动失败或超时会报告临时日志路径。live 的认证、网络或工具选择失败需要按实际错误处理。
- 8 秒是为了观察后台运行的教学窗口；真实模型响应很慢时可能错过窗口，可增加 researcher 中的等待时间后重跑，不应删除状态断言冒充已观察到异步执行。

清理成功时删除日志；失败时保留诊断日志。无需手动启动或关闭服务。下一步可回看 [同步子 Agent 的消息与文件边界](../../content/ch05-subagents.md)，区分进程内委派与服务端后台任务。